# Phase 2 HumanEval Planning Loop 분석 노트북

이 노트북은 `planning.yaml` 실행 결과로 생성되는 `step_logs.jsonl`, `trajectory_logs.jsonl`, `summary.json`, `failure_examples.json`을 분석

기본 결과 경로:

```text
results/phase2_qwen/humaneval/planning
```


## 1. 라이브러리 로드


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


## 2. 결과 디렉터리 설정

다른 실험 결과를 분석하려면 `OUTPUT_DIR`만 수정하세요.


In [ ]:
OUTPUT_DIR = Path("results/phase2_qwen/humaneval/planning")
ANALYSIS_DIR = OUTPUT_DIR / "analysis_reports"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

STEP_LOG_PATH = OUTPUT_DIR / "step_logs.jsonl"
TRAJ_LOG_PATH = OUTPUT_DIR / "trajectory_logs.jsonl"
SUMMARY_PATH = OUTPUT_DIR / "summary.json"
FAILURE_EXAMPLES_PATH = OUTPUT_DIR / "failure_examples.json"

print("OUTPUT_DIR:", OUTPUT_DIR)
print("ANALYSIS_DIR:", ANALYSIS_DIR)


OUTPUT_DIR: results/phase2_qwen/humaneval/repair
ANALYSIS_DIR: results/phase2_qwen/humaneval/repair/analysis_reports


## 3. 로더 함수


In [3]:
def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


## 4. 데이터 로드


In [4]:
steps = read_jsonl(STEP_LOG_PATH)
traj = read_jsonl(TRAJ_LOG_PATH)
summary = load_json(SUMMARY_PATH)
failure_examples = load_json(FAILURE_EXAMPLES_PATH)

print("steps:", steps.shape)
print("traj:", traj.shape)
print("summary keys:", list(summary.keys()))
print("failure example types:", len(failure_examples))

display(steps.head())
display(traj.head())


FileNotFoundError: [Errno 2] No such file or directory: 'results/phase2_qwen/humaneval/repair/step_logs.jsonl'

## 5. 전체 요약


In [ ]:
summary_df = pd.DataFrame([summary]).T.rename(columns={0: "value"})
display(summary_df)

summary_df.to_csv(ANALYSIS_DIR / "summary_table.csv")


## 6. 최종 성공률 및 실패 유형


In [ ]:
traj = traj.copy()
traj["passed"] = traj["final_status"].eq("PASS")

final_status_counts = traj["final_status"].value_counts(dropna=False)
failure_family_counts = traj["failure_family"].value_counts(dropna=False)

display(final_status_counts.rename("count").to_frame())
display(failure_family_counts.rename("count").to_frame())

final_status_counts.to_csv(ANALYSIS_DIR / "final_status_counts.csv")
failure_family_counts.to_csv(ANALYSIS_DIR / "failure_family_counts.csv")


### 최종 상태 분포 시각화


In [ ]:
ax = final_status_counts.plot(kind="bar", figsize=(10, 4), title="Final Status Counts")
ax.set_xlabel("final_status")
ax.set_ylabel("count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "final_status_counts.png", dpi=200)
plt.show()


## 7. Step-level 상태 분석


In [ ]:
step_status_counts = steps["status"].value_counts(dropna=False)
display(step_status_counts.rename("count").to_frame())

step_status_by_stage = pd.crosstab(steps["stage"], steps["status"])
display(step_status_by_stage)

step_status_counts.to_csv(ANALYSIS_DIR / "step_status_counts.csv")
step_status_by_stage.to_csv(ANALYSIS_DIR / "step_status_by_stage.csv")


### Step 상태 분포 시각화


In [ ]:
ax = step_status_counts.plot(kind="bar", figsize=(10, 4), title="Step Status Counts")
ax.set_xlabel("status")
ax.set_ylabel("count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "step_status_counts.png", dpi=200)
plt.show()


## 8. Call count별 성공률

`call_count`는 문제별로 실제 사용한 LLM 호출 수입니다.


In [ ]:
success_by_call_count = traj.groupby("call_count").agg(
    problems=("problem_id", "count"),
    success=("passed", "sum"),
    success_rate=("passed", "mean"),
    avg_tokens=("total_tokens", "mean"),
    avg_latency=("total_latency", "mean"),
).reset_index()

display(success_by_call_count)
success_by_call_count.to_csv(ANALYSIS_DIR / "success_by_call_count.csv", index=False)


In [ ]:
ax = success_by_call_count.plot(
    x="call_count",
    y="success_rate",
    kind="line",
    marker="o",
    figsize=(8, 4),
    title="Success Rate by Used Call Count",
)
ax.set_xlabel("call_count")
ax.set_ylabel("success_rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "success_rate_by_call_count.png", dpi=200)
plt.show()


## 9. First-pass attempt 분포

각 문제에서 처음 PASS가 나온 step을 계산합니다. `step_id=0`이면 initial generation에서 통과한 것입니다.


In [ ]:
pass_steps = steps[steps["status"].eq("PASS")].copy()

if len(pass_steps) > 0:
    first_pass = pass_steps.groupby("problem_id")["step_id"].min().rename("first_pass_step")
    first_pass_dist = first_pass.value_counts().sort_index().rename("count").to_frame()
    display(first_pass_dist)
    first_pass_dist.to_csv(ANALYSIS_DIR / "first_pass_attempt_distribution.csv")
else:
    first_pass = pd.Series(dtype=int)
    first_pass_dist = pd.DataFrame()
    print("No PASS steps found.")


In [ ]:
if len(first_pass_dist) > 0:
    ax = first_pass_dist["count"].plot(kind="bar", figsize=(8, 4), title="First PASS Step Distribution")
    ax.set_xlabel("first_pass_step")
    ax.set_ylabel("count")
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / "first_pass_step_distribution.png", dpi=200)
    plt.show()


## 10. Repair transition 분석

`transition_path`를 이용해 `TEST_FAIL -> PASS`, `EXEC_FAIL -> PASS` 같은 전이 빈도를 계산합니다.


In [ ]:
transitions = {}

for path in traj["transition_path"]:
    for a, b in zip(path, path[1:]):
        a_family = str(a).split(":")[0]
        b_family = str(b).split(":")[0]
        key = f"{a_family}->{b_family}"
        transitions[key] = transitions.get(key, 0) + 1

transition_df = pd.DataFrame(
    [{"transition": k, "count": v} for k, v in transitions.items()]
).sort_values("count", ascending=False)

display(transition_df)
transition_df.to_csv(ANALYSIS_DIR / "transition_counts.csv", index=False)


In [ ]:
if len(transition_df) > 0:
    ax = transition_df.set_index("transition")["count"].plot(
        kind="bar",
        figsize=(10, 4),
        title="Repair Transition Counts",
    )
    ax.set_xlabel("transition")
    ax.set_ylabel("count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / "transition_counts.png", dpi=200)
    plt.show()


## 11. Entropy 분석

`hf_model.generate()`가 entropy 값을 반환하면 step 로그에 저장됩니다. 없으면 0.0으로 fallback됩니다.


In [ ]:
entropy_cols = [
    "avg_entropy",
    "max_entropy",
    "entropy_std",
    "first_20pct_entropy",
    "last_20pct_entropy",
]
entropy_cols = [c for c in entropy_cols if c in steps.columns]

if entropy_cols:
    entropy_by_status = steps.groupby("status")[entropy_cols].mean().sort_values("avg_entropy", ascending=False)
    entropy_by_stage = steps.groupby("stage")[entropy_cols].mean()
    entropy_by_final = steps.merge(
        traj[["problem_id", "passed", "final_status"]],
        on="problem_id",
        how="left",
        suffixes=("", "_final"),
    ).groupby("passed")[entropy_cols].mean()

    display(entropy_by_status)
    display(entropy_by_stage)
    display(entropy_by_final)

    entropy_by_status.to_csv(ANALYSIS_DIR / "entropy_by_status.csv")
    entropy_by_stage.to_csv(ANALYSIS_DIR / "entropy_by_stage.csv")
    entropy_by_final.to_csv(ANALYSIS_DIR / "entropy_by_final_success.csv")
else:
    print("No entropy columns found.")


In [ ]:
if entropy_cols and "avg_entropy" in entropy_cols:
    ax = steps.boxplot(column="avg_entropy", by="status", figsize=(12, 5), rot=45)
    ax.set_title("Average Entropy by Step Status")
    ax.set_xlabel("status")
    ax.set_ylabel("avg_entropy")
    plt.suptitle("")
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / "avg_entropy_by_status_boxplot.png", dpi=200)
    plt.show()


## 12. 실패 문제 분석

최종 실패한 문제 중 호출 수, 토큰 수, latency가 큰 순서로 확인합니다.


In [ ]:
failed = traj[~traj["final_status"].eq("PASS")].copy()

hardest_failed = failed.sort_values(
    ["call_count", "total_tokens", "total_latency"],
    ascending=[False, False, False],
)[[
    "problem_id",
    "final_status",
    "failure_family",
    "call_count",
    "total_tokens",
    "total_latency",
    "transition_path",
]]

display(hardest_failed.head(30))
hardest_failed.to_csv(ANALYSIS_DIR / "hardest_failed_problems.csv", index=False)


## 13. 특정 문제 trajectory 자세히 보기

`PROBLEM_ID`를 바꿔서 개별 문제의 repair 흐름을 확인하세요.


In [ ]:
PROBLEM_ID = traj.iloc[0]["problem_id"]

problem_steps = steps[steps["problem_id"].eq(PROBLEM_ID)].copy()
problem_traj = traj[traj["problem_id"].eq(PROBLEM_ID)].copy()

display(problem_traj)
display(problem_steps[[
    "step_id",
    "stage",
    "status",
    "exec_ok",
    "test_pass",
    "error_type",
    "error_stage",
    "avg_entropy",
    "total_tokens",
    "latency_sec",
    "code_length",
]])


## 14. Failure examples 확인

`failure_examples.json`이 있으면 실패 유형별 대표 예시를 확인합니다.


In [ ]:
if failure_examples:
    print("failure types:", list(failure_examples.keys()))
    failure_type = next(iter(failure_examples.keys()))
    print("selected failure_type:", failure_type)
    display(pd.DataFrame([failure_examples[failure_type]]).T.rename(columns={0: "value"}))
else:
    print("No failure_examples.json found or file is empty.")


## 15. Markdown 리포트 생성


In [ ]:
report_lines = []
report_lines.append("# Repair Loop Analysis Report\n")

report_lines.append("## Summary\n")
for k, v in summary.items():
    report_lines.append(f"- **{k}**: {v}")

report_lines.append("\n## Final Status Counts\n")
report_lines.append(final_status_counts.rename("count").to_frame().to_markdown())

report_lines.append("\n## Failure Family Counts\n")
report_lines.append(failure_family_counts.rename("count").to_frame().to_markdown())

report_lines.append("\n## Success by Call Count\n")
report_lines.append(success_by_call_count.to_markdown(index=False))

report_lines.append("\n## Repair Transitions\n")
report_lines.append(transition_df.to_markdown(index=False))

report_lines.append("\n## Hardest Failed Problems\n")
report_lines.append(hardest_failed.head(20).to_markdown(index=False))

report_path = ANALYSIS_DIR / "repair_analysis_report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")

print("Saved markdown report:", report_path)


## 16. 저장된 분석 산출물 확인


In [ ]:
for path in sorted(ANALYSIS_DIR.glob("*")):
    print(path)
